# 10 - Models analysis

## Objectif :

Dans cette étape nous allons chercher à comprendre pourquoi les scoresdes différents models sont plafonnés à 0.50 et ensuite comprendre si il existe vraiment un signal ou juste du bruit dans les données d'entrainement

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, roc_auc_score

from src.data_loading import load_X_train, load_y_train
from src.modeling import build_logistic_regression_pipeline
from src.boosting_models import build_gradboosting_pipeline
from src.target import create_class_column
from src.features import ret_features

from src.validation import (
    create_expanding_window_folds, 
    check_temporal_folds
)

from src.evaluation import (
    evaluate_model_on_folds, 
    compare_model_results, 
    diagnostic_training
)

from src.advanced_boosting import (
    build_xgboost_pipeline, 
    build_lightgbm_pipeline 
)


In [3]:
X_train = load_X_train()
y_train = load_y_train()

In [4]:
df_train, _ = create_class_column(X_train, y_train)

In [11]:
dates = sorted(list(set(X_train["TS"])))
folds = create_expanding_window_folds(dates)
check_temporal_folds(folds,dates,validation_size=120)

True

In [5]:
reg_log = build_logistic_regression_pipeline()
gradient_boosting = build_gradboosting_pipeline()
xgboost = build_xgboost_pipeline()
lgbm = build_lightgbm_pipeline()

In [8]:
training_cols = ret_features(df_train)

In [9]:
models = [
    {
        "name": "logistic_regression",
        "estimator": reg_log
    },
    {
        "name": "gradient_boosting",
        "estimator": gradient_boosting
    },
    {
        "name": "xgboost",
        "estimator": xgboost
    },
    {
        "name": "lgbm",
        "estimator": lgbm
    }
]

In [ ]:
df_diagnostic = diagnostic_training(
    df_train,
    folds,
    models,
    training_cols
)

In [13]:
df_diagnostic

,ROW_ID,TS,fold,model,y_true,y_pred,y_proba,GROUP,ALLOCATION
0,428139,DATE_2043,1,logistic_regression,1,0,0.489712,3,ALLOCATION_09
1,428140,DATE_2043,1,logistic_regression,0,1,0.531128,3,ALLOCATION_101
2,428141,DATE_2043,1,logistic_regression,0,1,0.524725,3,ALLOCATION_103
3,428142,DATE_2043,1,logistic_regression,0,1,0.545132,3,ALLOCATION_104
4,428143,DATE_2043,1,logistic_regression,0,0,0.493621,3,ALLOCATION_107
...,...,...,...,...,...,...,...,...,...
395731,527068,DATE_2522,4,lgbm,0,1,0.517276,4,ALLOCATION_95
395732,527069,DATE_2522,4,lgbm,1,1,0.537359,3,ALLOCATION_96
395733,527070,DATE_2522,4,lgbm,1,1,0.536752,3,ALLOCATION_97
395734,527071,DATE_2522,4,lgbm,1,1,0.527804,4,ALLOCATION_98


In [14]:
df_diagnostic.to_csv("diagnostic.csv",index=False)

In [17]:
df_diagnostic[df_diagnostic["y_proba"] > 0.75]

,ROW_ID,TS,fold,model,y_true,y_pred,y_proba,GROUP,ALLOCATION
4591,432730,DATE_2069,1,logistic_regression,1,1,0.758113,3,ALLOCATION_116
25497,453636,DATE_2168,2,logistic_regression,1,1,0.977693,2,ALLOCATION_185
44524,472663,DATE_2264,2,logistic_regression,1,1,0.768575,1,ALLOCATION_01
52360,480499,DATE_2304,3,logistic_regression,1,1,0.817703,3,ALLOCATION_08
62569,490708,DATE_2352,3,logistic_regression,0,1,0.869246,2,ALLOCATION_185
76120,504259,DATE_2416,4,logistic_regression,1,1,0.769670,3,ALLOCATION_116
87204,515343,DATE_2471,4,logistic_regression,1,1,0.862628,2,ALLOCATION_185
95999,524138,DATE_2510,4,logistic_regression,0,1,0.765505,3,ALLOCATION_116


contruisons un tableau comparatif global et par fold

In [18]:
df_diagnostic["pred_is_correct"] = np.where(
    df_diagnostic["y_true"] == df_diagnostic["y_pred"],
    1,
    0
)

In [22]:
df_diagnostic.groupby(["model","fold"], as_index=False).agg(
   accuracy=("pred_is_correct","mean")
)

,model,fold,accuracy
0,gradient_boosting,1,0.519283
1,gradient_boosting,2,0.533694
2,gradient_boosting,3,0.524108
3,gradient_boosting,4,0.523889
4,lgbm,1,0.521108
5,lgbm,2,0.535908
6,lgbm,3,0.520877
7,lgbm,4,0.522876
8,logistic_regression,1,0.516588
9,logistic_regression,2,0.529923


Les résultats montrent une forte dépendance temporelle des performances. Les quatre modèles atteignent leur meilleure accuracy sur le fold 2, ce qui suggère que cette période contient un signal plus facilement exploitable. La régression logistique reste systématiquement en retrait, ce qui soutient l’intérêt des relations non linéaires. Toutefois, aucun modèle de boosting ne domine sur l’ensemble des folds : LightGBM arrive légèrement en tête sur les folds 1 et 2, tandis que Gradient Boosting est meilleur sur les folds 3 et 4. Les écarts étant très faibles, il serait prématuré de conclure à la supériorité robuste d’un modèle uniquement à partir de l’accuracy.

In [41]:
def evaluate_probabilities_by_fold(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:

    roc_auc_results = []
    log_loss_results = []

    models = df["model"].unique()
    folds = df["fold"].unique()

    for model in models:
        for fold in folds:

            df_fold = df[
                (df["model"] == model)
                & (df["fold"] == fold)
            ]

            roc_score = roc_auc_score(
                df_fold["y_true"],
                df_fold["y_proba"],
            )

            log_score = log_loss(
                df_fold["y_true"],
                df_fold["y_proba"],
            )

            roc_auc_results.append({
                "model": model,
                "fold": fold,
                "roc_auc": roc_score,
            })

            log_loss_results.append({
                "model": model,
                "fold": fold,
                "log_loss": log_score,
            })

    return (
        pd.DataFrame(roc_auc_results),
        pd.DataFrame(log_loss_results),
    )

In [38]:
roc , log = evaluate_probabilities_by_fold(df_diagnostic)

In [39]:
roc

,model,fold,roc_auc
0,logistic_regression,1,0.520935
1,logistic_regression,2,0.540789
2,logistic_regression,3,0.517971
3,logistic_regression,4,0.528017
4,gradient_boosting,1,0.519829
5,gradient_boosting,2,0.549852
6,gradient_boosting,3,0.523544
7,gradient_boosting,4,0.524227
8,xgboost,1,0.520011
9,xgboost,2,0.551208


In [40]:
log

,model,fold,log_loss
0,logistic_regression,1,0.692167
1,logistic_regression,2,0.690788
2,logistic_regression,3,0.692314
3,logistic_regression,4,0.691120
4,gradient_boosting,1,0.692187
5,gradient_boosting,2,0.691233
6,gradient_boosting,3,0.692039
7,gradient_boosting,4,0.691922
8,xgboost,1,0.692174
9,xgboost,2,0.690581


L’analyse probabiliste nuance les conclusions obtenues avec l’accuracy. Gradient Boosting conserve une légère avance en classification au seuil de 0,5, mais XGBoost obtient le meilleur ROC-AUC moyen et la meilleure log-loss moyenne, avec des résultats presque identiques à ceux de LightGBM. Les écarts entre les trois modèles de boosting restent néanmoins très faibles et ne permettent pas d’affirmer une domination robuste. Tous les modèles affichent leur meilleure performance sur le fold 2, ce qui suggère l’existence d’une période temporelle plus facilement prédictible. Enfin, les ROC-AUC systématiquement supérieurs à 0,5 indiquent la présence d’un signal de ranking hors échantillon, mais ce signal demeure faible.